# 复现 Fig. 9：暗物质晕的质量吸积史

这个 Notebook 负责论文 Fig. 9 的具体参数、绘图和数据导出。质量吸积公式本身仍由 `halohistory.py` 提供：

- `halohistory.calculate_mass_accretion_tracks`：计算不同现今质量 $M_0$ 对应的 $M(z)$；
- 本 Notebook：选择 Fig. 9 的红移网格和 13 个 $M_0$，绘制曲线并导出 CSV。

这样调整图例、坐标轴或图片大小时，不会改动主体物理模块。

## 1. 导入模块并定位项目目录

下面同时兼容从项目根目录或 `notebooks` 目录启动 Notebook 的情况。

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "halohistory.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import halohistory as hh

PROJECT_ROOT

WindowsPath('D:/pbh formation/pbh merge/2body channel and numerical simulation')

## 2. 设置 Fig. 9 的红移和现今质量

原图使用 $z=0$ 加上 49 个正红移点，一直到 $z=12$。$z=0$ 对保存完整质量轨迹很重要，但对数横轴不能显示零，因此绘图时只选取 `z > 0` 的点。

`PRESENT_DAY_MASSES_MSUN` 包含 $10^3,10^4,\ldots,10^{15} M_\odot$，每一行始终代表同一个今天质量 $M_0$ 的历史，不能在不同红移处重新排序。

In [2]:
redshift = np.linspace(0.0, 12.0, 50)
present_day_masses_msun = hh.PRESENT_DAY_MASSES_MSUN
positive_redshift = redshift > 0.0

present_day_masses_msun

array([1.e+03, 1.e+04, 1.e+05, 1.e+06, 1.e+07, 1.e+08, 1.e+09, 1.e+10,
       1.e+11, 1.e+12, 1.e+13, 1.e+14, 1.e+15])

## 3. 调用主体模块计算 $M(z)$

返回的 `mass_tracks_msun` 是二维数组：

- 行：13 个不同的现今质量 $M_0$；
- 列：50 个红移点；
- 元素：对应晕在该红移时的质量，单位为 $M_\odot$。

In [3]:
mass_tracks_msun = hh.calculate_mass_accretion_tracks(
    redshift,
    present_day_masses_msun=present_day_masses_msun,
)

print(f"质量轨迹数组形状：{mass_tracks_msun.shape}")

# 展示 M0=10^3、10^9、10^15 M_sun 三条轨迹在 z=0、约 6 和 12 的值。
mass_tracks_msun[[0, 6, 12]][:, [0, 24, 49]]

质量轨迹数组形状：(13, 50)


array([[1.00000000e+03, 2.15621323e+02, 3.80603708e+01],
       [1.00000000e+09, 8.53135941e+07, 5.26849207e+06],
       [1.00000000e+15, 4.54819070e+11, 1.08619032e+08]])

## 4. 绘制并保存 Fig. 9

横轴和纵轴都使用对数尺度。13 条曲线循环使用四种线型，图例中的指数来自每条轨迹的现今质量 $M_0$。

In [4]:
figure, axis = plt.subplots(figsize=(7.0, 4.8))
line_styles = ("-", "--", "-.", ":")

for index, mass0_msun in enumerate(present_day_masses_msun):
    exponent = int(np.log10(mass0_msun))
    axis.plot(
        redshift[positive_redshift],
        mass_tracks_msun[index, positive_redshift],
        linestyle=line_styles[index % len(line_styles)],
        label=rf"$M=10^{{{exponent}}}M_\odot$",
    )

axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlim(redshift[1], 12.0)
axis.set_ylim(1.0e1, 2.0e15)
axis.set_yticks(10.0 ** np.arange(3.0, 16.0, 3.0))
axis.set_xlabel(r"Redshift $z$")
axis.set_ylabel(r"$M(z)\;[M_\odot]$")
axis.legend(
    loc="upper right",
    fontsize=6.5,
    handlelength=2.7,
    labelspacing=0.25,
)

figure.tight_layout()
figure_path = PROJECT_ROOT / "fig9_reproduction.png"
figure.savefig(figure_path, dpi=220)
print(f"图片已保存：{figure_path}")
plt.show()

图片已保存：D:\pbh formation\pbh merge\2body channel and numerical simulation\fig9_reproduction.png


C:\Users\admin\AppData\Local\Temp\ipykernel_25324\2221404510.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. 导出 Fig. 9 的曲线数据

CSV 第一列为红移，后面 13 列依次对应 $M_0=10^3$ 到 $10^{15} M_\odot$。由于数组原本按“行对应质量、列对应红移”存放，写入 CSV 前需要转置。

In [ ]:
mass_columns = [
    f"mass_M0_{mass0_msun:.0e}_msun"
    for mass0_msun in present_day_masses_msun
]

csv_path = PROJECT_ROOT / "fig9_reproduction.csv"
np.savetxt(
    csv_path,
    np.column_stack((redshift, mass_tracks_msun.T)),
    delimiter=",",
    header=",".join(["z", *mass_columns]),
    comments="",
)

print(f"数据已保存：{csv_path}")

## 6. 当前复现边界

本图严格使用项目中已经实现的 Ludlow16/Appendix C 质量吸积参数。当前公式在高质量端会出现原论文 Fig. 9 没有显示的轨迹交叉，尤其涉及 $10^{13}$–$10^{15} M_\odot$。现有证据只说明论文原图在高质量端可能采用了未公开的参数处理，尚不能把这种差异解释成另一种已确认的物理模型。

因此 Notebook 保留固定 $M_0$ 的轨迹身份，不通过逐红移排序来人为消除交叉。